# I. Data Preprocessing

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import warnings, os
from pathlib import Path
warnings.filterwarnings('ignore')

DATA_PATH = Path('/kaggle/input/datasets/bnthanh/datathon-raw-data')

## 2. Load datasets

In [2]:

print("Loading all 14 datasets...\n")
sales       = pd.read_csv(DATA_PATH / 'sales.csv', parse_dates=['Date'])
orders      = pd.read_csv(DATA_PATH / 'orders.csv', parse_dates=['order_date'])
order_items = pd.read_csv(DATA_PATH / 'order_items.csv', low_memory=False)
products    = pd.read_csv(DATA_PATH / 'products.csv')
customers   = pd.read_csv(DATA_PATH / 'customers.csv', parse_dates=['signup_date'])
payments    = pd.read_csv(DATA_PATH / 'payments.csv')
shipments   = pd.read_csv(DATA_PATH / 'shipments.csv', parse_dates=['ship_date', 'delivery_date'])
returns     = pd.read_csv(DATA_PATH / 'returns.csv', parse_dates=['return_date'])
promotions  = pd.read_csv(DATA_PATH / 'promotions.csv', parse_dates=['start_date', 'end_date'])
geography   = pd.read_csv(DATA_PATH / 'geography.csv')
inventory   = pd.read_csv(DATA_PATH / 'inventory.csv', parse_dates=['snapshot_date'])
web_traffic = pd.read_csv(DATA_PATH / 'web_traffic.csv', parse_dates=['date'])
sample_sub  = pd.read_csv(DATA_PATH / 'sample_submission.csv', parse_dates=['Date'])
reviews     = pd.read_csv(DATA_PATH / 'reviews.csv', parse_dates=['review_date'])
print("Load datasets successfully!")

Loading all 14 datasets...

Load datasets successfully!


## 3. Audit Function 

In [3]:
# ==========================================
# AUDIT FUNCTION
# ==========================================
def audit_func(df, name):
    print(f"=== AUDIT REPORT: {name.upper()} ===")
    
    # 1. Basic Stats
    print(f"1. Shape: {df.shape}")
    dups = df.duplicated().sum()
    print(f"2. Duplicates: {dups} ({dups/len(df)*100:.2f}%)")
    
    # 2. Missing Values
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) > 0:
        print("3. Missing Values:")
        for col, val in missing.items():
            print(f"   - {col}: {val} rows ({val/len(df)*100:.2f}%)")
    else:
        print("3. Missing Values: None")
        
    # 3. Logical Checks (Business Rules)
    print("4. Logical Anomalies:")
    has_logic_error = False
    
    if name == 'Sales':
        neg_sales = len(df[(df['Revenue'] < 0) | (df['COGS'] < 0)])
        if neg_sales > 0: print(f"   - [!] {neg_sales} rows with negative Revenue/COGS."); has_logic_error = True
            
    elif name == 'Promotions':
        inv_dates = len(df[df['end_date'] < df['start_date']])
        if inv_dates > 0: print(f"   - [!] {inv_dates} promos with end_date < start_date."); has_logic_error = True
            
    elif name == 'Products':
        neg_price = len(df[(df['price'] < 0) | (df['cogs'] < 0)])
        if neg_price > 0: print(f"   - [!] {neg_price} products with negative price/cogs."); has_logic_error = True
            
    elif name == 'Shipments':
        inv_ship = len(df[df['delivery_date'] < df['ship_date']])
        if inv_ship > 0: print(f"   - [!] {inv_ship} shipments with delivery_date < ship_date."); has_logic_error = True
            
    elif name == 'Inventory':
        inv_fill = len(df[(df['fill_rate'] < 0) | (df['fill_rate'] > 1)])
        if inv_fill > 0: print(f"   - [!] {inv_fill} records with fill_rate out of [0,1] bounds."); has_logic_error = True
            
    elif name == 'Web Traffic':
        inv_bounce = len(df[(df['bounce_rate'] < 0) | (df['bounce_rate'] > 1)])
        if inv_bounce > 0: print(f"   - [!] {inv_bounce} records with bounce_rate out of bounds."); has_logic_error = True
            
    # CHECK LOGIC CHO REVIEWS (Giả sử cột điểm là 'rating' với thang điểm 1-5)
    elif name == 'Reviews':
        if 'rating' in df.columns:
            inv_rating = len(df[(df['rating'] < 1) | (df['rating'] > 5)])
            if inv_rating > 0: print(f"   - [!] {inv_rating} reviews with rating out of [1,5] bounds."); has_logic_error = True

    if not has_logic_error:
        print("   - None detected.")
        
    print("-" * 50 + "\n")


Run Audit Func

In [4]:
all_datasets = {'Sales': sales, 'Order Items': order_items, 'Products': products, 
                'Shipments': shipments, 'Promotions': promotions, 'Inventory': inventory, 
                'Web Traffic': web_traffic, 'Orders': orders, 'Customers': customers, 
                'Payments': payments, 'Returns': returns, 'Geography': geography, 
                'Reviews': reviews} 

# Run the comprehensive audit
for name, df in all_datasets.items():
    audit_func(df, name)

=== AUDIT REPORT: SALES ===
1. Shape: (3833, 3)
2. Duplicates: 0 (0.00%)
3. Missing Values: None
4. Logical Anomalies:
   - None detected.
--------------------------------------------------

=== AUDIT REPORT: ORDER ITEMS ===
1. Shape: (714669, 7)
2. Duplicates: 0 (0.00%)
3. Missing Values:
   - promo_id: 438353 rows (61.34%)
   - promo_id_2: 714463 rows (99.97%)
4. Logical Anomalies:
   - None detected.
--------------------------------------------------

=== AUDIT REPORT: PRODUCTS ===
1. Shape: (2412, 8)
2. Duplicates: 0 (0.00%)
3. Missing Values: None
4. Logical Anomalies:
   - None detected.
--------------------------------------------------

=== AUDIT REPORT: SHIPMENTS ===
1. Shape: (566067, 4)
2. Duplicates: 0 (0.00%)
3. Missing Values: None
4. Logical Anomalies:
   - None detected.
--------------------------------------------------

=== AUDIT REPORT: PROMOTIONS ===
1. Shape: (50, 10)
2. Duplicates: 0 (0.00%)
3. Missing Values:
   - applicable_category: 40 rows (80.00%)
4. Logical 

### Short results analysis
- Bảng promotions (Cột applicable_category thiếu 80%)
Ngữ nghĩa kinh doanh: Trong ngành bán lẻ, khi một chương trình khuyến mãi không ghi rõ áp dụng cho ngành hàng (category) nào, điều đó ngầm hiểu là nó áp dụng cho toàn bộ cửa hàng (Store-wide). Ví dụ: "Giảm 10% toàn sàn nhân ngày Black Friday".

Tác động đến Mô hình: Nếu điền giá trị All_Categories vào các ô trống này, mô hình sẽ học được sự khác biệt giữa một đợt sale "cục bộ" (chỉ giảm giá áo thun) và một đợt sale "toàn diện" (giảm mọi thứ).

- Bảng order_items (Cột promo_id thiếu 61.34%, promo_id_2 thiếu 99.97%)
Ngữ nghĩa kinh doanh: * promo_id rỗng (61.34%) đơn giản nghĩa là khách hàng mua món đồ đó với giá gốc (không dùng mã giảm giá).

promo_id_2 rỗng (99.97%) nghĩa là tính năng "áp dụng 2 mã giảm giá cùng lúc" (stacking promo) cực kỳ hiếm người dùng (chỉ 0.03% dùng).

Cách xử lý: * Với promo_id: Điền chữ NO_PROMO. Khi thực hiện LEFT JOIN bảng này với bảng promotions, những dòng NO_PROMO sẽ không khớp với bất kỳ mã nào, sinh ra giá trị NaN, và sẽ fill các giá trị NaN đó bằng 0 (tức là mức giảm giá = 0).

Với promo_id_2: Vì thiếu đến 99.97%, cột này mang rất ít thông tin (Information Gain thấp) nhưng lại làm nhiễu mô hình -> chuyển nó thành một cột đánh dấu (Cờ - Flag) has_stacked_promo (1 nếu có dùng mã 2, 0 nếu không), sau đó xóa cột gốc đi.



### Fix data base on results

In [5]:
# ==========================================
# 1. FIX PROMOTIONS
# ==========================================
# Datetime format
promotions['start_date'] = pd.to_datetime(promotions['start_date'])
promotions['end_date'] = pd.to_datetime(promotions['end_date'])

# Process applicable_category (Missing 80%) -> Filled "All_Categories"
promotions['applicable_category'] = promotions['applicable_category'].fillna('All_Categories')

# Logic err: end date < start date
invalid_promos = len(promotions[promotions['end_date'] < promotions['start_date']])
if invalid_promos > 0:
    print(f"[Promotions] Reverted {invalid_promos} row has End Date < Start Date.")
    mask = promotions['end_date'] < promotions['start_date']
    promotions.loc[mask, ['start_date', 'end_date']] = promotions.loc[mask, ['end_date', 'start_date']].values

# ==========================================
# 2. FIX ORDER_ITEMS
# ==========================================
# Process promo_id (Missing 61.34%) -> Filled "NO_PROMO"
order_items['promo_id'] = order_items['promo_id'].fillna('NO_PROMO')

# Process promo_id_2 (Missing 99.97%) -> Convert to (Flag) then Drop
order_items['has_stacked_promo'] = order_items['promo_id_2'].notna().astype(int)
order_items = order_items.drop(columns=['promo_id_2'])
print("[Order Items] Processed NaN for promo_id and convert promo_id_2 to flag.")

# ==========================================
# 3. FIX SALES
# ==========================================
sales['Date'] = pd.to_datetime(sales['Date'])
# Get absolute value for Revenue/COGS (negative)
sales['Revenue'] = sales['Revenue'].apply(lambda x: abs(x) if x < 0 else x)
sales['COGS'] = sales['COGS'].apply(lambda x: abs(x) if x < 0 else x)
# Remove outliers using IQR 1% - 99%
Q1, Q3 = sales['Revenue'].quantile(0.01), sales['Revenue'].quantile(0.99)
sales['Revenue'] = sales['Revenue'].clip(lower=Q1, upper=Q3)
print("[Sales] date formatted, processed negative values and cut outliers.")

# ==========================================
# 4. FIX WEB TRAFFIC
# ==========================================
web_traffic['date'] = pd.to_datetime(web_traffic['date'])
# Processed Bounce_rate over [0, 1]
web_traffic['bounce_rate'] = web_traffic['bounce_rate'].clip(lower=0, upper=1)
print("[Web Traffic] Formatted & Bounce_rate.")

[Order Items] Processed NaN for promo_id and convert promo_id_2 to flag.
[Sales] date formatted, processed negative values and cut outliers.
[Web Traffic] Formatted & Bounce_rate.


## 4. Check data! 

In [6]:
print("--- AUTOMATED DATA INTEGRITY CHECK ---")

try:
    # 1. MISSING VALUES VERIFICATION
    for name, df in all_datasets.items():
        if name == 'Order Items':
             assert df['promo_id'].isnull().sum() == 0, "Assertion Error: 'Order Items' table contains null values in 'promo_id'."
        elif name == 'Promotions':
             assert df['applicable_category'].isnull().sum() == 0, "Assertion Error: 'Promotions' table contains null values in 'applicable_category'."
        elif name == 'Reviews' and 'review_text' in df.columns:
             assert df['review_text'].isnull().sum() == 0, "Assertion Error: 'Reviews' table contains null values in 'review_text'."
             
    # 2. BUSINESS LOGIC VERIFICATION
    assert len(sales[(sales['Revenue'] < 0) | (sales['COGS'] < 0)]) == 0, "Logic Error: Negative values detected in Sales (Revenue or COGS)."
    assert len(products[(products['price'] < 0) | (products['cogs'] < 0)]) == 0, "Logic Error: Negative values detected in Products (price or cogs)."
    
    assert len(promotions[promotions['end_date'] < promotions['start_date']]) == 0, "Logic Error: end_date is prior to start_date in Promotions."
    assert len(shipments[shipments['delivery_date'] < shipments['ship_date']]) == 0, "Logic Error: delivery_date is prior to ship_date in Shipments."
    
    assert inventory['fill_rate'].max() <= 1.0 and inventory['fill_rate'].min() >= 0.0, "Logic Error: fill_rate bounds violation. Expected [0.0, 1.0]."
    assert web_traffic['bounce_rate'].max() <= 1.0 and web_traffic['bounce_rate'].min() >= 0.0, "Logic Error: bounce_rate bounds violation. Expected [0.0, 1.0]."
    
    if 'rating' in reviews.columns:
        assert reviews['rating'].max() <= 5 and reviews['rating'].min() >= 1, "Logic Error: rating bounds violation. Expected [1, 5]."

    print("[INFO] STATUS: PASSED.")
    print("[INFO] MESSAGE: All data integrity assertions have been successfully validated.")
    print("[INFO] NEXT STEP: Data is ready for Feature Engineering and Master Table Aggregation.")

except AssertionError as e:
    print(f"[ERROR] STATUS: FAILED.")
    print(f"[ERROR] REASON: {e}")
    print("[WARN] ACTION REQUIRED: Review the data cleansing process (Cell 2) and execute it again before proceeding.")

--- AUTOMATED DATA INTEGRITY CHECK ---
[INFO] STATUS: PASSED.
[INFO] MESSAGE: All data integrity assertions have been successfully validated.
[INFO] NEXT STEP: Data is ready for Feature Engineering and Master Table Aggregation.


## 5. Export preprocessed data

In [7]:
import os

print("--- EXPORTING ALL PREPROCESSED DATA ---")

# 1. Create a directory for the output data
output_dir = '/kaggle/working/preprocessed'
os.makedirs(output_dir, exist_ok=True)

# 2. Loop through and automatically export all 14 tables
for name, df in all_datasets.items():
    # Standardize file names (e.g., 'Order Items' -> 'clean_order_items.csv')
    clean_name = name.lower().replace(' ', '_')
    file_name = f"clean_{clean_name}.csv"
    file_path = os.path.join(output_dir, file_name)
    
    # Export to CSV format (excluding the index column)
    df.to_csv(file_path, index=False)
    
    # Print professional logs for each exported file
    print(f"[INFO] Exported: {file_name:<25} | Shape: {df.shape}")

print("-" * 50)
print(f"[SUCCESS] All 14 data tables have been successfully exported to: {output_dir}/")
print("[INFO] Ready to hand over for the Feature Engineering or Modeling phase!")

--- EXPORTING ALL PREPROCESSED DATA ---
[INFO] Exported: clean_sales.csv           | Shape: (3833, 3)
[INFO] Exported: clean_order_items.csv     | Shape: (714669, 8)
[INFO] Exported: clean_products.csv        | Shape: (2412, 8)
[INFO] Exported: clean_shipments.csv       | Shape: (566067, 4)
[INFO] Exported: clean_promotions.csv      | Shape: (50, 10)
[INFO] Exported: clean_inventory.csv       | Shape: (60247, 17)
[INFO] Exported: clean_web_traffic.csv     | Shape: (3652, 7)
[INFO] Exported: clean_orders.csv          | Shape: (646945, 8)
[INFO] Exported: clean_customers.csv       | Shape: (121930, 7)
[INFO] Exported: clean_payments.csv        | Shape: (646945, 4)
[INFO] Exported: clean_returns.csv         | Shape: (39939, 7)
[INFO] Exported: clean_geography.csv       | Shape: (39948, 4)
[INFO] Exported: clean_reviews.csv         | Shape: (113551, 7)
--------------------------------------------------
[SUCCESS] All 14 data tables have been successfully exported to: /kaggle/working/preproc